# 第 4 周练习 — 代码生成器（Python → C++）

## 练习目标（理念）

用前沿模型把**慢速 Python**重写为**高性能 C++**，然后**编译并基准测试**，用数据证明加速。两套模型各自生成 C++（`gpt-4o-mini` 与 `claude-haiku-4.5`），方便对比——第 4 周主题正是「选择正确的模型」。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模型对比 | OpenAI vs Anthropic 同题生成 |
| system prompt 约束输出 | 只要纯 C++，不要 Markdown |
| 编译 / 基准（Benchmark） | `g++ -O3` + `time` 计时 |
| 正确性核对 | 输出数字必须与 Python 一致 |

## 怎么跑

1. `.env` 配好 OpenAI / Anthropic 密钥
2. 本机需有 `g++`（Windows 上脚本含 MinGW 回退路径）
3. 从上到下运行：先跑 Python 基线，再对两个模型生成 → 编译 → 计时


In [1]:
# ========== 导入与环境：双客户端 + 定位 g++ 编译器 ==========

# os / shutil / subprocess / time：路径、找可执行文件、跑编译、计时
import os, shutil, subprocess, time
# anthropic：Claude Messages API 客户端
import anthropic
# OpenAI：Chat Completions 客户端（这里叫 frontier）
from openai import OpenAI
# load_dotenv：从 .env 读入密钥
from dotenv import load_dotenv

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)
# 默认读 OPENAI_API_KEY 的云端客户端
frontier = OpenAI()
# Anthropic 官方客户端（读 ANTHROPIC_API_KEY）
claude = anthropic.Anthropic()

# 优先 PATH 里的 g++；找不到则回退到常见 MinGW 安装路径（Windows）
GPP = shutil.which("g++") or r"C:\ProgramData\mingw64\mingw64\bin\g++.exe"
# 中间产物目录：生成的 .cpp / .exe 写到这里
BUILD = r"C:\Users\Public\llm_week4_build"
# 确保构建目录存在
os.makedirs(BUILD, exist_ok=True)
# 打印实际用到的编译器路径，方便排查
print("g++:", GPP)


g++: C:\ProgramData\mingw64\mingw64\bin\g++.exe


In [2]:
# ========== 待加速的慢速 Python：数素数（纯整数，便于与 C++ 输出逐字比对）==========

# 三引号字符串：整段当作「源程序文本」发给模型，也会被 exec 计时
PYTHON = '''
def count_primes(n):
    count = 0
    for x in range(2, n):
        d, prime = 2, True
        while d * d <= x:
            if x % d == 0:
                prime = False
                break
            d += 1
        if prime:
            count += 1
    return count

print(count_primes(300000))
'''


In [3]:
# ========== 生成 C++、清洗围栏、编译运行计时 ==========

# system：要求只回 C++17 源码且输出与 Python 一致（英文 prompt 保持原样）
SYSTEM = (
    "Rewrite the given Python as a single high-performance C++17 program. "
    "Reply with ONLY C++ source code - no markdown, no commentary. Include all headers "
    "and an int main() that prints exactly the same output as the Python."
)

# 去掉模型可能加的 ``` 代码围栏行
def clean(text):
    return "\n".join(l for l in text.splitlines() if not l.strip().startswith("```")).strip()

# 按 provider 分流：Anthropic Messages vs OpenAI Chat Completions
def to_cpp(provider, model):
    if provider == "anthropic":
        # Claude：system 单独参数；user 内容就是 PYTHON 源码
        msg = claude.messages.create(model=model, system=SYSTEM, max_tokens=1500,
                                     messages=[{"role": "user", "content": PYTHON}])
        return clean(msg.content[0].text)
    # OpenAI 兼容路径：system + user 两条 messages
    r = frontier.chat.completions.create(model=model, messages=[
        {"role": "system", "content": SYSTEM}, {"role": "user", "content": PYTHON}])
    return clean(r.choices[0].message.content)

# 写文件 → g++ -O3 编译 → 运行并返回 (stdout, 秒数)
def compile_and_run(cpp, name):
    # 源文件与可执行文件路径（Windows 下用 .exe 后缀）
    src, exe = os.path.join(BUILD, name + ".cpp"), os.path.join(BUILD, name + ".exe")
    # 写出模型生成的 C++
    with open(src, "w") as f:
        f.write(cpp)
    # -O3 开优化；-std=c++17；失败则 check=True 抛错
    subprocess.run([GPP, "-O3", "-std=c++17", "-o", exe, src], check=True, capture_output=True, text=True)
    # 只给「运行阶段」计时（不含编译）
    t = time.time()
    out = subprocess.run([exe], capture_output=True, text=True).stdout.strip()
    return out, time.time() - t


### 基线：运行 Python

先在本机 `exec` 同一段 `PYTHON`，记下耗时 `py_time`，后面用来算加速比（speedup）。


In [4]:
# ========== Python 基线计时：exec 同一段源码字符串 ==========

# 空命名空间字典：exec 的全局环境（避免污染笔记本全局）
ns = {}
# 记录 exec 前后时间差，得到 Python 耗时
t = time.time(); exec(PYTHON, ns); py_time = time.time() - t
# 打印基线秒数，便于和后面 C++ 对比
print(f"python: {py_time:.3f}s")


25997
python: 0.996s


### 用每个模型生成 C++，再编译、运行、比较

对 OpenAI 与 Anthropic 各跑一轮：生成 → 编译 → 运行 → 打印输出与加速比。输出数字应与 Python 基线一致。


In [5]:
# ========== 多模型对比循环：生成 → 编译运行 → 算 speedup ==========

# 每项：(展示标签, provider 分支名, 实际 model id)——id 字符串保持原样
for label, provider, model in [
    ("gpt-4o-mini", "openai", "gpt-4o-mini"),
    ("claude-haiku-4.5", "anthropic", "claude-haiku-4-5-20251001"),
]:
    # 调用对应 API，得到清洗后的 C++ 源码
    cpp = to_cpp(provider, model)
    # 文件名去掉点号，避免奇怪路径；拿到输出与耗时
    out, cpp_time = compile_and_run(cpp, label.replace(".", ""))
    # 加速比 = Python 时间 / C++ 时间；除零时视为无穷大
    speedup = py_time / cpp_time if cpp_time else float("inf")
    # 对齐打印：标签、输出、耗时、加速倍数
    print(f"{label:<18} output={out:<8} time={cpp_time:.4f}s  speedup x{speedup:,.0f}")


gpt-4o-mini        output=25997    time=0.0710s  speedup x14


claude-haiku-4.5   output=25997    time=0.0715s  speedup x14
